# 🎤 Maya1 Voice Engine for Eva
**Free voice cloning with emotions — runs on Colab's free GPU**

1. Click **Runtime → Change runtime type → T4 GPU**
2. Click **Run All** (Runtime → Run all)
3. Wait ~3 minutes for model to load
4. Copy the **public URL** at the bottom
5. Paste it in Eva's Admin Dashboard

⚠️ Colab disconnects after ~90 min idle. Re-run when needed.

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers snac soundfile gradio numpy

In [ ]:
# Step 2: Load Maya1 model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC
import soundfile as sf
import numpy as np
import tempfile
import gradio as gr

# Token IDs for Maya1
CODE_START_TOKEN_ID = 128257
CODE_END_TOKEN_ID = 128258
CODE_TOKEN_OFFSET = 128266
SNAC_MIN_ID = 128266
SNAC_MAX_ID = 156937
SNAC_TOKENS_PER_FRAME = 7
SOH_ID = 128259
EOH_ID = 128260
SOA_ID = 128261
BOS_ID = 128000
TEXT_EOT_ID = 128009

print('Loading Maya1 model (3B params)...')
model = AutoModelForCausalLM.from_pretrained(
    'maya-research/maya1',
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained('maya-research/maya1', trust_remote_code=True)
print(f'Maya1 loaded! Vocab: {len(tokenizer)} tokens')

print('Loading SNAC audio decoder...')
snac_model = SNAC.from_pretrained('hubertsiuzdak/snac_24khz').eval().to('cuda')
print('Ready! 🎤')

In [ ]:
# Step 3: Voice generation functions

def build_prompt(description, text):
    soh = tokenizer.decode([SOH_ID])
    eoh = tokenizer.decode([EOH_ID])
    soa = tokenizer.decode([SOA_ID])
    sos = tokenizer.decode([CODE_START_TOKEN_ID])
    eot = tokenizer.decode([TEXT_EOT_ID])
    bos = tokenizer.bos_token
    formatted = f'<description="{description}"> {text}'
    return soh + bos + formatted + eot + eoh + soa + sos

def extract_snac_codes(token_ids):
    try: eos_idx = token_ids.index(CODE_END_TOKEN_ID)
    except ValueError: eos_idx = len(token_ids)
    return [t for t in token_ids[:eos_idx] if SNAC_MIN_ID <= t <= SNAC_MAX_ID]

def unpack_snac(snac_tokens):
    if snac_tokens and snac_tokens[-1] == CODE_END_TOKEN_ID:
        snac_tokens = snac_tokens[:-1]
    frames = len(snac_tokens) // SNAC_TOKENS_PER_FRAME
    snac_tokens = snac_tokens[:frames * SNAC_TOKENS_PER_FRAME]
    if frames == 0: return [[], [], []]
    l1, l2, l3 = [], [], []
    for i in range(frames):
        s = snac_tokens[i*7:(i+1)*7]
        l1.append((s[0] - CODE_TOKEN_OFFSET) % 4096)
        l2.extend([(s[1] - CODE_TOKEN_OFFSET) % 4096, (s[4] - CODE_TOKEN_OFFSET) % 4096])
        l3.extend([(s[2] - CODE_TOKEN_OFFSET) % 4096, (s[3] - CODE_TOKEN_OFFSET) % 4096,
                   (s[5] - CODE_TOKEN_OFFSET) % 4096, (s[6] - CODE_TOKEN_OFFSET) % 4096])
    return [l1, l2, l3]

def generate_voice(text, description, emotion_tags=''):
    """Generate speech with voice description and optional emotion tags."""
    if emotion_tags:
        text = text  # User adds tags inline like <laugh> or <whisper>
    
    prompt = build_prompt(description, text)
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to('cuda') for k, v in inputs.items()}
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            min_new_tokens=28,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=CODE_END_TOKEN_ID,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    generated = outputs[0, inputs['input_ids'].shape[1]:].tolist()
    snac_tokens = extract_snac_codes(generated)
    
    if len(snac_tokens) < 7:
        return None
    
    levels = unpack_snac(snac_tokens)
    codes = [torch.tensor(l, dtype=torch.long, device='cuda').unsqueeze(0) for l in levels]
    
    with torch.inference_mode():
        z_q = snac_model.quantizer.from_codes(codes)
        audio = snac_model.decoder(z_q)[0, 0].cpu().numpy()
    
    if len(audio) > 2048:
        audio = audio[2048:]
    
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        sf.write(f.name, audio, 24000)
        return f.name

print('Voice generation ready! ✅')

In [ ]:
# Step 4: Launch Gradio API (Eva connects to this)

VOICE_PRESETS = {
    'Eva Calm (Female)': 'Female, in her late 20s with an Indian accent. Warm, soft, calm tone. Low pitch, gentle timbre, slow pacing.',
    'Eva Motivator (Female)': 'Female, in her 30s with an American accent. Energetic, powerful, inspiring. Normal pitch, clear diction, fast pacing.',
    'Eva Therapist (Female)': 'Female, in her 30s with a warm British accent. Empathetic, thoughtful, professional. Medium pitch, warm timbre, measured pacing.',
    'Eva Companion (Female)': 'Female, in her 20s with an American accent. Casual, friendly, warm. Normal pitch, conversational pacing.',
    'Eva Lullaby (Female)': 'Female, in her 30s with a soft Indian accent. Dreamy, whisper-like, very slow. Low pitch, airy timbre, very slow pacing.',
    'Eva Deep (Male)': 'Male, in his 40s with a deep British accent. Warm, wise, calm. Low pitch, rich timbre, slow pacing.',
    'Custom': '',
}

def tts_api(text, voice_preset, custom_description, emotions):
    desc = custom_description if voice_preset == 'Custom' else VOICE_PRESETS[voice_preset]
    if not desc:
        desc = 'Female, in her late 20s with an Indian accent. Warm, calm tone.'
    if emotions:
        text = emotions + ' ' + text
    output = generate_voice(text, desc)
    return output

demo = gr.Interface(
    fn=tts_api,
    inputs=[
        gr.Textbox(label='Text', placeholder='What should Eva say?', lines=3),
        gr.Dropdown(choices=list(VOICE_PRESETS.keys()), value='Eva Calm (Female)', label='Voice Preset'),
        gr.Textbox(label='Custom Voice Description (only if Custom selected)', placeholder='e.g., Male, 30s, Indian accent, warm tone...'),
        gr.Textbox(label='Emotion Tags (optional)', placeholder='e.g., <laugh> or <whisper> or <sigh>'),
    ],
    outputs=gr.Audio(label='Generated Speech'),
    title='🎤 Eva Voice Engine (Maya1)',
    description='Powered by Maya1 — 20+ emotions, voice design, production quality. Free on Colab GPU.',
)

# Launch with public URL so Eva can call it
demo.queue()
demo.launch(share=True, show_error=True)
# ☝️ Copy the PUBLIC URL and paste in Eva Admin Dashboard